# Notebook 04: Sensitivity Analysis

**Objective:** Identify which SPICE parameters most strongly influence the I-V curves.

We cover:
1. Sobol' variance-based global sensitivity
2. Morris screening method
3. Parameter ranking for extraction priority
4. Visualize sensitivity indices

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt

from src.device.mosfet import MOSFETLevel3, MOSFETParamsLevel3
from src.device.curves import generate_iv_curves
from src.extraction.sensitivity import (
    SobolAnalyzer, MorrisAnalyzer, ParameterBounds
)
from src.viz.plots import plot_sensitivity_heatmap

## 1. Define the Sensitivity Problem

We measure how much each parameter affects the **log-space RMSE** between the default and perturbed I-V curves.

In [ ]:
# Baseline model
p_default = MOSFETParamsLevel3()
base_model = MOSFETLevel3(p_default)
base_vg, _ = generate_iv_curves(base_model)

# Define parameter bounds for sensitivity analysis
bounds = ParameterBounds(
    names=["VTH0", "GAMMA", "U0", "THETA", "VSAT", "ETA0", "LAMBDA", "N0"],
    bounds=[
        (0.2, 0.8), (0.2, 1.0),
        (200, 600), (0.02, 0.15),
        (6e6, 14e6), (0.02, 0.12),
        (0.02, 0.10), (1.2, 2.5),
    ],
)

def cost_fn(x):
    """RMSE in log-Id space between perturbed and baseline."""
    p = MOSFETParamsLevel3(
        VTH0=float(x[0]), GAMMA=float(x[1]),
        U0=float(x[2]), THETA=float(x[3]),
        VSAT=float(x[4]), ETA0=float(x[5]),
        LAMBDA=float(x[6]), N0=float(x[7]),
    )
    m = MOSFETLevel3(p)
    vg, _ = generate_iv_curves(m)
    ids_log = np.log10(np.maximum(base_vg.ids, 1e-12))
    sim_log = np.log10(np.maximum(vg.ids, 1e-12))
    return float(np.sqrt(np.mean((ids_log - sim_log) ** 2)))

print(f"Problem: {bounds.n_params} parameters, bounds defined")
print(f"Baseline cost: {cost_fn([0.45, 0.4, 400, 0.05, 10e6, 0.05, 0.05, 1.5]):.6e}")

## 2. Sobol' Sensitivity Analysis

Decomposes output variance into first-order (S1) and total-effect (ST) indices.

In [ ]:
analyzer = SobolAnalyzer(cost_fn, bounds)
result = analyzer.analyze(n_base=256, verbose=True)

# Visualize
fig = plot_sensitivity_heatmap(result, "Sobol Sensitivity — MOSFET Level-3")
plt.show()

## 3. Parameter Ranking

Ranking by total-effect index (ST) tells us which parameters to prioritize in extraction.

In [ ]:
ranking = sorted(
    zip(result["param_names"], result["S1"], result["ST"]),
    key=lambda x: -x[2]
)

print(f"{'Rank':<6s} {'Parameter':<12s} {'S1':>8s} {'ST':>8s} {'Interaction':>12s}")
print("-" * 50)
for rank, (name, s1, st) in enumerate(ranking, 1):
    interaction = st - s1
    bar = "#" * int(st * 50)
    print(f"{rank:<6d} {name:<12s} {s1:>8.4f} {st:>8.4f} {interaction:>12.4f} {bar}")

## 4. Interpretation

- **High S1, low ST-S1:** Parameter has a strong *direct* (linear/additive) effect
- **High ST, large ST-S1:** Parameter has strong *interaction* effects (nonlinear)
- **Low S1, low ST:** Parameter has negligible influence — can be fixed at nominal values

### Extraction Strategy from Sensitivity
- Fix **low-ST** parameters at nominal values to reduce dimensionality
- Start extraction with **high-ST** parameters (most influential)
- Use tighter bounds for **high-ST** parameters in global search

## 5. Morris Screening (Quick Alternative)

Morris method is faster than Sobol' and good for initial screening with many parameters.

In [ ]:
morris = MorrisAnalyzer(cost_fn, bounds)
mr = morris.analyze(n_trajectories=10, verbose=True)

# Plot mu* vs sigma
fig, ax = plt.subplots(figsize=(8, 6))
for i, name in enumerate(mr["param_names"]):
    ax.scatter(mr["mu_star"][i], mr["sigma"][i], s=100, label=name)
    ax.annotate(name, (mr["mu_star"][i], mr["sigma"][i]),
                textcoords="offset points", xytext=(5, 5), fontsize=8)

ax.set_xlabel("mu* (Mean Absolute Effect)")
ax.set_ylabel("sigma (Std Dev of Effect)")
ax.set_title("Morris Screening: mu* vs sigma")
ax.axhline(0, color='gray', linestyle='--', alpha=0.3)
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Summary

- **Sobol'** provides rigorous variance decomposition but requires many evaluations
- **Morris** is a cheaper screening method for initial parameter ranking
- **Top influencers** (high ST) should be prioritized in extraction
- **Negligible parameters** can be fixed to reduce optimization dimensionality
- This analysis directly guides the parameter extraction strategy